# Reasoning Models এবং GRPO

**PART A --** GRPO-এর কৌশল, আলাদা করে সরাসরি মাপা, তারপর একটি খেলনা **VERIFIABLE**-reward task-এ end to end চালানো। প্রতিটি "prompt" একটি target digit-sum (যেমন "0-5 এর মধ্যে 3টি digit খুঁজুন যাদের যোগফল 7"); "policy" প্রতি prompt-এ একটি 3-digit sequence sample করে; reward **DETERMINISTIC** এবং প্রকৃতই যাচাইযোগ্য: digit-গুলোর যোগফল আসলেই target হলে 1.0, নইলে 0.0 -- একটি আসল correctness check, "extract করা চূড়ান্ত উত্তর কি ground truth-এর সাথে মিলেছে" (README section 1)-এর প্রতিনিধি, Lesson 3-এর ইচ্ছাকৃতভাবে exploit-যোগ্য "hack token" reward-এর বিপরীতে।

তিনটি জিনিস সততার সাথে মাপা হয়, এই নির্দিষ্ট খেলনা task-এর কাঠিন্যের উপর কতটা নির্ভর করে সেই ক্রম অনুযায়ী:

1. একটি বৈশিষ্ট্য যা task নির্বিশেষে **সবসময়** সত্য: GRPO-এর group-relative advantage ব্যর্থ completion-গুলোকে একটি আসল negative ("এটা থেকে দূরে সরো") signal দেয়; শুধু raw-reward ("no baseline") কখনোই দেয় না।
2. একটি বৈশিষ্ট্য যা **সবসময়** সত্য, z-scoring-এর সংজ্ঞা থেকেই সরাসরি প্রমাণযোগ্য: prompt যত সহজ বা কঠিনই হোক, GRPO-এর advantage মোটামুটি একটি **স্থির** signal scale বজায় রাখে, অথচ raw reward-এর signal-শক্তি খুব সহজ বা খুব কঠিন prompt-এর জন্য প্রায় শূন্যে নেমে যায় (variance = p(1-p), যা p -> 0 বা 1 হলে বিলীন হয়)।
3. একটি end-to-end training run যা দেখায় পূর্ণ GRPO update (group-relative advantage + clipped surrogate + একটি frozen reference-এর বিপরীতে KL penalty) আসলেই খেলনা task-টি সমাধান করে, সাথে একটি সৎ মন্তব্য যে কেন এই **নির্দিষ্ট** ছোট, low-dimensional খেলনা search space-কে শেষ পর্যন্ত সমাধান করতে স্থূলতর পদ্ধতিরও বৈশিষ্ট্য 1-2-এর সুবিধা লাগে না -- action space ও reasoning length এক লেসনের runtime-এ যা আঁটে তার চেয়ে অনেক বড় হলে সেই বৈশিষ্ট্যগুলোই গুরুত্বপূর্ণ হয়।

**PART B --** Test-time scaling: একটি **ইচ্ছাকৃতভাবে** আংশিক-train করা (অর্থাৎ এখনো অসম্পূর্ণ) policy ব্যবহার করে, প্রতি prompt-এ N-টি completion sample করা, প্রতিটির ফলাফল-যোগফল extract করা, এবং সেই N-টি extract করা উত্তরের উপর একটি **MAJORITY VOTE** নেওয়া -- হুবহু Phase 07 Lesson 2-এর self-consistency কৌশল, এখানে একটি স্থির, ইতিমধ্যে-train করা model-এ অতিরিক্ত inference compute খরচ করার সুচিন্তিত উপায় হিসেবে প্রয়োগ করা। N বাড়ার সাথে আসল accuracy মাপা হয় এবং diminishing-returns বক্ররেখাটি সরাসরি দেখা যায়।

**Runtime:** CPU-তে ~15-30 সেকেন্ড (কোনো আসল neural network নেই -- Lesson 3-এর মতো, এটি সরাসরি একটি ছোট logits tensor optimize করে, যাতে প্রতিটি RL কৌশল সম্পূর্ণ স্বচ্ছ থাকে)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। প্রথম কয়েকটি section শুধু setup (task, constants, GRPO-র building block); Part A ও Part B-এর demo তাদের নিজ নিজ cell-এর শেষে চলে।

In [ ]:
import random
from collections import Counter

import torch
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## 0. খেলনা verifiable-reward task ও hyperparameter

`NUM_PROMPTS`-টি ভিন্ন "problem", প্রতিটি `SEQ_LEN`-টি digit (0..`VOCAB_SIZE`-1) চায় যাদের যোগফল একটি নির্দিষ্ট target। এখানে group size, PPO-ধাঁচের clip range, KL penalty weight, learning rate ও iteration সংখ্যাও সংজ্ঞায়িত — এখানে কোনো demo চলে না।

In [ ]:
# ---------------------------------------------------------------------------
# খেলনা verifiable-reward task: NUM_PROMPTS-টি ভিন্ন "problem," প্রতিটি
# SEQ_LEN-টি digit (0..VOCAB_SIZE-1) চায় যাদের যোগফল একটি নির্দিষ্ট target।
# ---------------------------------------------------------------------------

VOCAB_SIZE = 6                      # digit 0-5
SEQ_LEN = 3                         # প্রতি response-এ 3টি digit
TARGET_SUMS = [4, 6, 9, 11]         # প্রতি prompt-এ একটি target (সর্বোচ্চ সম্ভাব্য যোগফল = 15)
NUM_PROMPTS = len(TARGET_SUMS)

GROUP_SIZE = 16          # প্রতি rollout-এ প্রতি prompt-এ sample করা G-টি completion (GRPO-এর "group")
CLIP_EPS = 0.2           # Lesson 3-এর হুবহু একই PPO trust-region clip range
KL_BETA = 0.03           # frozen reference policy-র বিপরীতে KL penalty weight
PPO_EPOCHS = 4           # প্রতি rollout batch-এ পুনর্ব্যবহৃত update epoch, Lesson 3-এর মতোই
LR = 0.04
NUM_ITERS = 120
PARTIAL_TRAIN_ITERS = 20  # ইচ্ছাকৃতভাবে আগেভাগে থামানো -- Part B দেখুন

## 1. Reference policy, verifiable reward ও rollout

`make_reference_logits()` frozen reference policy `pi_ref` তৈরি করে (একটি SFT শুরুবিন্দুর প্রতিনিধি)। `verifiable_reward()` একটি deterministic checker — digit-গুলোর যোগফল target হলে 1.0, নইলে 0.0। `rollout()` প্রতিটি prompt-এর জন্য একসাথে `group_size`-টি completion sample করে এবং actions, old/ref log-prob ও raw reward ফেরত দেয়।

In [ ]:
def make_reference_logits():
    """Frozen reference policy pi_ref (README section 2/4) -- প্রতি prompt-এ
    সামান্য non-uniform, যেকোনো reasoning-RL fine-tuning-এর আগের একটি SFT
    শুরুবিন্দুর প্রতিনিধি।"""
    torch.manual_seed(7)
    return torch.randn(NUM_PROMPTS, SEQ_LEN, VOCAB_SIZE) * 0.3


REFERENCE_LOGITS = make_reference_logits()


def verifiable_reward(actions):
    """Verifiable reward (README section 1): prompt p-এর sample করা digit-গুলোর
    যোগফল আসলেই TARGET_SUMS[p] হলে 1.0, নইলে 0.0। একটি deterministic
    checker, learned model নয় -- Lesson 3-এর reward-model পরিবর্তের তুলনায়
    প্রকৃতই game করা অসম্ভব, কারণ exploit করার মতো কোনো proxy নেই।"""
    targets = torch.tensor(TARGET_SUMS).view(1, NUM_PROMPTS)
    sums = actions.sum(dim=-1)                      # (batch, NUM_PROMPTS)
    return (sums == targets).float()


def rollout(policy_logits, group_size=GROUP_SIZE):
    """প্রতিটি prompt-এর জন্য একসাথে `group_size`-টি completion sample করে।
    ফেরত দেয় actions (group, NUM_PROMPTS, SEQ_LEN), old/ref log-prob (group,
    NUM_PROMPTS, SEQ_LEN), এবং raw verifiable reward (group, NUM_PROMPTS)।"""
    with torch.no_grad():
        probs = F.softmax(policy_logits, dim=-1)                     # (NUM_PROMPTS, SEQ_LEN, V)
        dist = torch.distributions.Categorical(
            probs=probs.unsqueeze(0).expand(group_size, -1, -1, -1)
        )
        actions = dist.sample()                                      # (group, NUM_PROMPTS, SEQ_LEN)
        old_log_probs = dist.log_prob(actions)                       # (group, NUM_PROMPTS, SEQ_LEN)

        ref_probs = F.softmax(REFERENCE_LOGITS, dim=-1)
        ref_dist = torch.distributions.Categorical(
            probs=ref_probs.unsqueeze(0).expand(group_size, -1, -1, -1)
        )
        ref_log_probs = ref_dist.log_prob(actions)

        raw_reward = verifiable_reward(actions)                      # (group, NUM_PROMPTS)
    return actions, old_log_probs, ref_log_probs, raw_reward

## 2. Advantage: GRPO-এর group-relative বনাম no-baseline ablation

`group_relative_advantage()` হলো GRPO-এর পুরো কৌশল: প্রতিটি prompt-এর G-টি reward-কে সেই prompt-এরই group mean/std দিয়ে normalize করা — কোনো critic network নেই, কোনো global baseline নেই। `no_baseline_advantage()` হলো ablation: একই raw reward, কোনো baseline বাদ না দিয়ে — ফলে ব্যর্থ completion কখনো নিচে ঠেলা হয় না।

In [ ]:
def group_relative_advantage(raw_reward):
    """GRPO-এর পুরো কৌশল (README section 2-3): প্রতিটি prompt-এর G-টি
    reward-কে সেই PROMPT-এরই group mean/std দিয়ে normalize করা -- কোনো critic
    network নেই, কোনো global baseline নেই। সেই completion-এর প্রতিটি token
    position-এ broadcast করা হয়।"""
    mean = raw_reward.mean(dim=0, keepdim=True)               # (1, NUM_PROMPTS)
    std = raw_reward.std(dim=0, keepdim=True)
    advantage = (raw_reward - mean) / (std + 1e-4)             # (group, NUM_PROMPTS)
    return advantage.unsqueeze(-1).expand(-1, -1, SEQ_LEN)     # SEQ_LEN জুড়ে broadcast


def no_baseline_advantage(raw_reward):
    """Ablation: একই raw reward, কোনো baseline একেবারেই বাদ না দিয়ে --
    প্রতিটি ব্যর্থ completion (reward=0) হুবহু শূন্য advantage দেয়, তাই এটিকে
    কখনো সক্রিয়ভাবে নিচে ঠেলা যায় না, শুধু সফলগুলোকে উপরে ঠেলা হয়
    (README section 4-এর ablation)।"""
    return raw_reward.unsqueeze(-1).expand(-1, -1, SEQ_LEN)

## 3. Clipped update ও training loop

`clipped_update()` হলো Lesson 3-এর `ppo_update`-এর হুবহু একই clipped-surrogate-plus-KL-penalty update, বাড়তি `NUM_PROMPTS` মাত্রার উপর সাধারণীকৃত; GRPO ও ablation-এর মধ্যে শুধু পাঠানো `advantage` tensor-টি ভিন্ন। `train()` reference logits থেকে শুরু করে বারবার rollout → advantage → update চালায় এবং প্রতি iteration-এর success rate-এর history ফেরত দেয়।

In [ ]:
def clipped_update(policy_logits, actions, old_log_probs, ref_log_probs, advantage, optimizer,
                    group_size=GROUP_SIZE):
    """Lesson 3-এর ppo_update-এর হুবহু একই clipped-surrogate-plus-KL-penalty
    update, বাড়তি NUM_PROMPTS মাত্রার উপর সাধারণীকৃত। GRPO ও ablation-এর
    মধ্যে শুধু পাঠানো `advantage` tensor-টি ভিন্ন।"""
    for _ in range(PPO_EPOCHS):
        probs = F.softmax(policy_logits, dim=-1)
        dist = torch.distributions.Categorical(
            probs=probs.unsqueeze(0).expand(group_size, -1, -1, -1)
        )
        new_log_probs = dist.log_prob(actions)

        ratio = torch.exp(new_log_probs - old_log_probs)
        surrogate1 = ratio * advantage
        surrogate2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantage
        policy_loss = -torch.min(surrogate1, surrogate2).mean()

        kl_term = (new_log_probs - ref_log_probs.detach()).mean()   # sample করা KL(pi||pi_ref) estimate
        loss = policy_loss + KL_BETA * kl_term

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def train(use_group_baseline, label, num_iters=NUM_ITERS, seed=0, verbose=True):
    torch.manual_seed(seed)
    policy_logits = REFERENCE_LOGITS.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([policy_logits], lr=LR)

    history = []
    for it in range(1, num_iters + 1):
        actions, old_log_probs, ref_log_probs, raw_reward = rollout(policy_logits.detach())

        advantage = group_relative_advantage(raw_reward) if use_group_baseline \
            else no_baseline_advantage(raw_reward)

        clipped_update(policy_logits, actions, old_log_probs, ref_log_probs, advantage, optimizer)

        success_rate = raw_reward.mean().item()
        history.append(success_rate)
        if verbose and (it % 20 == 0 or it == 1):
            print(f"  [{label}] iter {it:4d}   success rate (avg over {GROUP_SIZE * NUM_PROMPTS} "
                  f"rollouts) = {success_rate:.3f}")

    return policy_logits.detach(), history

## 4. Demo 1 ও Demo 2-এর measurement function

- **Demo 1:** negative-advantage কৌশলটি মূর্ত করা — প্রায়-random শুরু থেকে একটি rollout নিয়ে GRPO ও no-baseline-এর অধীনে কত ভাগ completion-এর advantage কঠোরভাবে negative তা মাপা।
- **Demo 2:** সহজ/মাঝারি/কঠিন prompt জুড়ে signal-scale-এর সামঞ্জস্য। বিশুদ্ধ পরিসংখ্যান -- simulated Bernoulli(p) group, কোনো আসল training run থেকে স্বাধীন, তাই সবসময় একইভাবে পুনরুৎপাদিত হয়।

এই cell শুধু function সংজ্ঞায়িত করে; এগুলো পরের section-এর `part_a_demo()` থেকে call হয়।

In [ ]:
# ---------------------------------------------------------------------------
# DEMO 1: negative-advantage কৌশলটি, মূর্তভাবে
# ---------------------------------------------------------------------------

def measure_negative_advantage_fraction():
    actions, _, _, raw_reward = rollout(REFERENCE_LOGITS.clone())
    grpo_adv = group_relative_advantage(raw_reward)[:, :, 0]
    noB_adv = no_baseline_advantage(raw_reward)[:, :, 0]
    grpo_neg_frac = (grpo_adv < 0).float().mean().item()
    noB_neg_frac = (noB_adv < 0).float().mean().item()
    return grpo_neg_frac, noB_neg_frac


# ---------------------------------------------------------------------------
# DEMO 2: সহজ/মাঝারি/কঠিন prompt জুড়ে signal-scale-এর সামঞ্জস্য। বিশুদ্ধ
# পরিসংখ্যান -- simulated Bernoulli(p) group, কোনো আসল training run থেকে
# স্বাধীন, তাই উপরের policy যেভাবেই train হোক না কেন, এটি সবসময় একইভাবে
# পুনরুৎপাদিত হয়।
# ---------------------------------------------------------------------------

def measure_signal_consistency():
    torch.manual_seed(0)
    LARGE_GROUP = 64
    NUM_SIMULATED_GROUPS = 4000
    ps = [0.03, 0.125, 0.5, 0.875, 0.97]   # খুব কঠিন -> মাঝারি -> খুব সহজ
    results = []
    for p in ps:
        rewards = torch.bernoulli(torch.full((NUM_SIMULATED_GROUPS, LARGE_GROUP), p))
        raw_std = rewards.std().item()
        mean = rewards.mean(dim=1, keepdim=True)
        std = rewards.std(dim=1, keepdim=True)
        grpo_adv = (rewards - mean) / (std + 1e-4)
        grpo_std = grpo_adv.std().item()
        results.append((p, raw_std, grpo_std))
    return results

## 5. Part A: GRPO-এর কৌশল, সরাসরি মাপা, তারপর end to end চালানো

তিনটি ধাপ: (1) negative-advantage কৌশল, (2) সহজ/মাঝারি/কঠিন prompt জুড়ে signal-scale-এর সামঞ্জস্য, এবং (3) পূর্ণ GRPO update দিয়ে end-to-end training, একই reward/clipping/KL penalty কিন্তু কোনো group baseline ছাড়া একটি তুলনামূলক run-সহ। Demo-টি cell-এর শেষে চলে।

In [ ]:
def part_a_demo():
    print("=" * 78)
    print("PART A: GRPO'S MECHANISM, MEASURED DIRECTLY, THEN RUN END TO END")
    print("=" * 78)
    print(f"{NUM_PROMPTS} prompts, each: find {SEQ_LEN} digits in 0..{VOCAB_SIZE - 1} summing to a target.")
    print(f"Targets: {TARGET_SUMS}   Group size (completions sampled per prompt): {GROUP_SIZE}")
    print("Reward is a DETERMINISTIC correctness check -- 1.0 if digits sum to the")
    print("target, 0.0 otherwise. No learned reward model, no human labels at RL time.\n")

    print("-" * 78)
    print("1. NEGATIVE-ADVANTAGE MECHANISM (one rollout from the near-random start)")
    print("-" * 78)
    grpo_neg_frac, noB_neg_frac = measure_negative_advantage_fraction()
    print(f"Fraction of sampled completions with a STRICTLY NEGATIVE advantage:")
    print(f"  GRPO (group-relative baseline):  {grpo_neg_frac:.1%}")
    print(f"  No baseline (raw reward only):   {noB_neg_frac:.1%}")
    print("-> With no baseline, a failed completion (reward=0) gets advantage exactly")
    print("   0 -- it is never actively pushed down, only successes get pushed up.")
    print("   GRPO's group-relative advantage is NEGATIVE for below-group-average")
    print("   completions (which includes every failure whenever ANY group member")
    print("   succeeded), giving the policy a real 'move away from this' signal that")
    print("   plain REINFORCE-without-a-baseline structurally cannot provide.")

    print("\n" + "-" * 78)
    print("2. SIGNAL-SCALE CONSISTENCY ACROSS EASY/MEDIUM/HARD PROMPTS")
    print("-" * 78)
    print("(Pure statistics on simulated Bernoulli(p) groups -- independent of any")
    print("actual training run below, so this always reproduces identically.)\n")
    results = measure_signal_consistency()
    print(f"{'per-sample success prob p':28}{'raw-reward advantage std':>28}{'GRPO advantage std':>22}")
    for p, raw_std, grpo_std in results:
        print(f"{p:<28}{raw_std:>28.3f}{grpo_std:>22.3f}")
    raw_ratio = max(r[1] for r in results) / min(r[1] for r in results)
    grpo_ratio = max(r[2] for r in results) / min(r[2] for r in results)
    print(f"\n-> Raw-reward advantage's scale varies {raw_ratio:.1f}x between the hardest/easiest")
    print(f"   prompts and the p=0.5 prompt (it literally IS sqrt(p*(1-p)), which vanishes")
    print(f"   toward both extremes) -- meaning a very easy or very hard prompt gives the")
    print(f"   policy almost no gradient signal at all, even though there is still plenty")
    print(f"   left to learn (e.g. going from 95% to 99.9% success). GRPO's group-relative")
    print(f"   z-scoring keeps the advantage's scale within {grpo_ratio:.1f}x across the SAME")
    print(f"   sweep -- a consistently strong learning signal regardless of how easy or")
    print(f"   hard any individual prompt happens to be.")

    print("\n" + "-" * 78)
    print("3. END-TO-END: DOES THE FULL GRPO UPDATE ACTUALLY SOLVE THE TASK?")
    print("-" * 78)
    trained_policy_grpo, history_grpo = train(use_group_baseline=True, label="GRPO")

    print("\n" + "(for comparison -- same reward, same clipping, same KL penalty, only")
    print(" the advantage computation differs: no group baseline at all)")
    _, history_noB = train(use_group_baseline=False, label="no-baseline")

    final_grpo = sum(history_grpo[-10:]) / 10
    final_noB = sum(history_noB[-10:]) / 10
    print(f"\nFinal success rate (last 10 rollouts): GRPO={final_grpo:.1%}   no-baseline={final_noB:.1%}")
    print("-> Both methods solve this SPECIFIC toy task -- it is a tiny, low-dimensional")
    print("   search space (216 possible digit sequences per prompt), and Adam's own")
    print("   adaptive step sizing partly compensates for raw reward's inconsistent scale")
    print("   here. That is an honest property of this toy example, not a claim that")
    print("   baselines don't matter: demos 1-2 above measure the ACTUAL mechanism GRPO")
    print("   provides directly, and that mechanism is what separates 'learns from sparse,")
    print("   verifiable rewards at all' from 'doesn't' once the search space is a real")
    print("   space of token sequences many steps long, not 216 three-digit combinations.")

    return trained_policy_grpo


part_a_demo()

## 6. Part B: Test-time scaling -- N sample-এর উপর majority vote, বনাম N

একটি নতুন policy মাত্র `PARTIAL_TRAIN_ITERS` GRPO iteration-এর পর ইচ্ছাকৃতভাবে আগেভাগে থামানো হয় — একটি প্রকৃতই অসম্পূর্ণ policy, ঠিক যেখানে অতিরিক্ত inference compute খরচ করা সার্থক। প্রতিটি trial-এ একটি random prompt-এর জন্য N-টি completion sample করে, প্রতিটির digit-sum-কে "উত্তর" ধরে majority vote নেওয়া হয় (Phase 07 Lesson 2-এর self-consistency), এবং N বাড়ার সাথে accuracy মাপা হয়। Demo-টি cell-এর শেষে চলে।

In [ ]:
# ===========================================================================
# PART B: TEST-TIME SCALING -- N SAMPLE-এর উপর MAJORITY VOTE, বনাম N
# ===========================================================================

def sample_completions(policy_logits, prompt_idx, n):
    probs = F.softmax(policy_logits[prompt_idx], dim=-1)          # (SEQ_LEN, V)
    dist = torch.distributions.Categorical(probs=probs.unsqueeze(0).expand(n, -1, -1))
    actions = dist.sample()                                        # (n, SEQ_LEN)
    return actions.sum(dim=-1).tolist()                            # extract করা "উত্তর" (যোগফল)


def majority_vote_accuracy(policy_logits, n, num_trials=3000):
    correct = 0
    for _ in range(num_trials):
        prompt_idx = random.randrange(NUM_PROMPTS)
        extracted_sums = sample_completions(policy_logits, prompt_idx, n)
        majority_answer = Counter(extracted_sums).most_common(1)[0][0]
        if majority_answer == TARGET_SUMS[prompt_idx]:
            correct += 1
    return correct / num_trials


def part_b_demo():
    print("\n" + "=" * 78)
    print("PART B: TEST-TIME SCALING -- MAJORITY VOTE OVER N SAMPLES")
    print("=" * 78)
    print(f"Training a FRESH policy but stopping deliberately early, after only")
    print(f"{PARTIAL_TRAIN_ITERS} GRPO iterations (Part A trains to {NUM_ITERS}) -- a genuinely")
    print("IMPERFECT, still-learning policy, exactly the regime where spending extra")
    print("INFERENCE compute is worth it (a policy already at ~100% one-shot accuracy")
    print("has nothing left for majority voting to improve).\n")
    partial_policy, partial_history = train(
        use_group_baseline=True, label="partial-GRPO", num_iters=PARTIAL_TRAIN_ITERS, verbose=False
    )
    one_shot_acc = sum(partial_history[-5:]) / 5
    print(f"One-shot (N=1) success rate after {PARTIAL_TRAIN_ITERS} iterations: {one_shot_acc:.1%}\n")

    print("For each trial: sample N completions for a random prompt, extract each one's")
    print("digit-sum as its 'answer,' and take the MAJORITY-VOTE answer across all N --")
    print("exactly Phase 07 Lesson 2 section 3's self-consistency mechanism, now spent as")
    print("the deliberate way a reasoning model uses extra INFERENCE compute on a fixed,")
    print("already-trained policy (as opposed to Phase 03 Lesson 5's scaling laws, which")
    print("spend more compute at TRAINING time instead).\n")

    ns = [1, 3, 5, 9, 15, 25]
    print(f"{'N (samples per vote)':25}{'majority-vote accuracy':>26}")
    accuracies = []
    for n in ns:
        acc = majority_vote_accuracy(partial_policy, n)
        accuracies.append(acc)
        print(f"{n:<25}{acc:>26.1%}")

    print(f"\n-> Accuracy rises from {accuracies[0]:.1%} at N=1 (a single sample) toward "
          f"{accuracies[-1]:.1%}")
    print(f"   at N={ns[-1]}, with shrinking marginal gains per extra sample -- the same")
    print(f"   diminishing-returns shape Phase 07 Lesson 2 section 4's Condorcet Jury")
    print(f"   Theorem predicts for any per-sample accuracy above chance. Because each")
    print(f"   WRONG digit-sequence can land on several different incorrect sums (not")
    print(f"   one single 'the' wrong answer), this is really the multi-candidate")
    print(f"   plurality-voting regime from Lesson 2 section 5, not the plain binary")
    print(f"   case -- wrong votes split against each other across several wrong sums,")
    print(f"   which is part of why the correct answer wins the vote more easily than")
    print(f"   the raw per-sample accuracy alone would suggest.")


part_b_demo()

## সবগুলো demo একসাথে

`main()` Part A ও Part B দুটি demo-ই এক চালে পরপর চালায়, ঠিক মূল script-এর মতো। উপরের cell-গুলোতে demo-গুলো ইতিমধ্যে চলেছে, তাই দুবার না চালানোর জন্য call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    part_a_demo()
    part_b_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন